# FedCrohn — Severity-Aware Extension

Multi-task GAT predicting Crohn's diagnosis **and** SES-CD severity, federated across 4 real hospitals (HMP2 host transcriptomics).

**Run order:** Setup cells 1-7 first, then any experiment cell.

Outputs below are from separate sessions (Kaggle Quick Save keeps only the last-executed cell's output), merged here into one notebook.

## Setup — run cells 1-7 in order

In [3]:
# Cell 1 — Setup
import sys, os

PROJECT_PATH = "/kaggle/input/datasets/bhanavi1231/hmp2-severity"
sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

print("CWD:", os.getcwd())
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
import numpy as np, scipy
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)

CWD: /kaggle/input/datasets/bhanavi1231/hmp2-severity
GPU available: False
GPU: None
numpy: 2.0.2
scipy: 1.16.3


In [4]:
# Cell 2 — Verify files & imports
import os, pickle

def load_pkl(path):
    with open(path, 'rb') as f:
        content = f.read()
    return pickle.loads(content.replace(b'\r\n', b'\n'), encoding='latin1')

for f in [
    "marshalledP3/totGeneSet.m.min0",
    "marshalledP3/adj_cache.pkl",
    "phenopediaCrohnGenes/CrohnGenes.txt",
    "host_tx_counts.tsv",
    "hmp2_metadata_2018-08-20.csv",
    "hmp2_loader.py",
    "severity_model.py",
    "run_severity_fl.py",
]:
    exists = os.path.exists(f)
    size = f"{os.path.getsize(f)/1e6:.1f} MB" if exists else "MISSING"
    print(f"{'OK' if exists else 'MISSING':8s} {size:10s}  {f}")

print("\nTesting imports...")
from sources.GATmodel import GATCrohnModel
from sources.buildGeneGraph import build_adj_from_string, build_adj_phenopedia
from sources.FedExplainer import LocalExplainer, GlobalExplainer
import sources.GraphConv as GCN
print("All imports OK")

gs = load_pkl("marshalledP3/totGeneSet.m.min0")
print(f"Gene set size: {len(gs)}")

OK       0.0 MB      marshalledP3/totGeneSet.m.min0
OK       1.9 MB      marshalledP3/adj_cache.pkl
OK       0.0 MB      phenopediaCrohnGenes/CrohnGenes.txt
OK       38.7 MB     host_tx_counts.tsv
OK       9.1 MB      hmp2_metadata_2018-08-20.csv
OK       0.0 MB      hmp2_loader.py
OK       0.0 MB      severity_model.py
OK       0.0 MB      run_severity_fl.py

Testing imports...
All imports OK
Gene set size: 691


In [5]:
# Cell 3 — Load gene graph from cache
from sources.buildGeneGraph import build_adj_from_string, build_adj_phenopedia
from sources.readPhenopedia import readPhenopedia
import numpy as np

geneList = sorted(load_pkl("marshalledP3/totGeneSet.m.min0"))
weightPhenoPGenes, _ = readPhenopedia("phenopediaCrohnGenes/CrohnGenes.txt")

# string_db/ is intentionally NOT in this dataset (631 MB, unused).
# build_adj_from_string returns the cached adjacency before touching it.
adj = build_adj_from_string(
    geneList,
    string_links_path="string_db/9606.protein.links.v12.0.txt",
    string_info_path="string_db/9606.protein.info.v12.0.txt",
    threshold=700,
    cache_path="marshalledP3/adj_cache.pkl"
)

if adj is None or np.array_equal(adj, np.eye(len(geneList))):
    print("Falling back to Phenopedia-based adjacency")
    adj = build_adj_phenopedia(geneList, weightPhenoPGenes)

print(f"Adjacency shape: {adj.shape}")
print(f"Non-zero edges : {int((adj > 0).sum() - adj.shape[0])}")
print(f"Density        : {(adj > 0).mean():.4f}")

Found 691 associated genes.
Loading adjacency matrix from cache: marshalledP3/adj_cache.pkl
Adjacency shape: (691, 691)
Non-zero edges : 6390
Density        : 0.0148


In [6]:
# Cell 4 — Imports & globals
import numpy as np
import torch as t
from collections import OrderedDict

from sources.readPhenopedia import readPhenopedia
from sources import utils as U
from sources.GATmodel import GATCrohnModel
from sources.FedExplainer import LocalExplainer, GlobalExplainer

DEVICE = t.device("cuda" if t.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

MODEL_DIR = "/kaggle/working/models"
os.makedirs(MODEL_DIR, exist_ok=True)

import random as _random
_random.seed(42); np.random.seed(42)
t.manual_seed(42); t.cuda.manual_seed_all(42)
print("Random seeds set to 42")

Using device: cpu
Random seeds set to 42


In [7]:
# Cell 5 — Patch GATLayer: memory-efficient per-head attention
# Peak alloc [B,N,N] per head instead of [B,N,N,H] all at once.
# B=4, N=691: ~7 MB per head vs ~3.6 GB.
import torch
import torch.nn.functional as F
from sources.GATmodel import GATLayer, GATCrohnModel

def _efficient_gat_forward(self, x, adj):
    B, N, _ = x.shape
    h = self.W(x).view(B, N, self.num_heads, self.out_features)

    head_outputs, last_alpha = [], None
    for head_idx in range(self.num_heads):
        h_head = h[:, :, head_idx, :]
        a_head = self.a[head_idx]
        h_i = h_head.unsqueeze(2).expand(B, N, N, self.out_features)
        h_j = h_head.unsqueeze(1).expand(B, N, N, self.out_features)
        e = self.leakyrelu(torch.cat([h_i, h_j], dim=-1).matmul(a_head))
        mask = (adj == 0).unsqueeze(0)
        e = e.masked_fill(mask, float('-inf'))
        alpha = F.softmax(e, dim=2)
        alpha = self.dropout(alpha)
        head_outputs.append(torch.bmm(alpha, h_head))
        last_alpha = alpha.detach()
        del h_i, h_j, e, alpha
        torch.cuda.empty_cache()

    out = torch.stack(head_outputs, dim=0).mean(dim=0)
    return out, last_alpha

def _fixed_get_gene_importance(self):
    if self.attention_weights is None:
        return None
    return self.attention_weights.mean(dim=0).sum(dim=0)

GATLayer.forward = _efficient_gat_forward
GATCrohnModel.get_gene_importance = _fixed_get_gene_importance
print("GATLayer patched — memory-efficient per-head attention")

GATLayer patched — memory-efficient per-head attention


In [8]:
# Cell 6 — Attach severity heads + DP
#
# WARNING: run this cell exactly once per session. It wraps __init__,
# and re-running double-wraps it. Restart the kernel to redo.
import sys
from severity_model import (attach_severity_heads, patch_wrapper_multitask,
                            fedavg_multitask, severity_metrics, diagnosis_metrics)
from sources.GraphConv import NNwrapper

if hasattr(GATCrohnModel, "_severity_attached"):
    del GATCrohnModel._severity_attached

attach_severity_heads(GATCrohnModel, n_sev_levels=4, pool_genes=False)
GATCrohnModel._severity_attached = True
print("severity heads attached — CORAL, 4 levels, FLATTEN")

patch_wrapper_multitask(NNwrapper, DEVICE, dp_sigma=1.5, dp_per_example=True)
print("NNwrapper patched — DP ON, dp_sigma=1.5, per-example clipping")

severity heads attached — CORAL, 4 levels, FLATTEN
NNwrapper patched — DP ON, dp_sigma=1.5, per-example clipping


In [9]:
# Cell 7 — Load HMP2 severity data
from hmp2_loader import build_hmp2_dataset, make_node_features

data = build_hmp2_dataset(
    meta_path   = "hmp2_metadata_2018-08-20.csv",
    counts_path = "host_tx_counts.tsv",
    geneList    = geneList,
    n_levels    = 4,
)

host_transcriptomics samples : 252
  diagnosis  CD/UC/nonIBD    : 127/74/51
  severity-labelled (CD only): 119  across 40 participants
  severity level counts      : {0: 58, 1: 26, 2: 25, 3: 10}

  per-site (all samples / severity-labelled):
    Cedars-Sinai       78 /  41
    Cincinnati         90 /  40
    MGH                60 /  28
    MGH Pediatrics     24 /  10

expression matrix : 55765 genes x 254 samples
  geneList coverage: 657/691 (95.1%)
  dropped 1 sample(s) with metadata but no expression column: ['HSM9JTBX']

final cohort      : 251 samples
  severity-labelled: 118
  severity levels  : {0: 58, 1: 26, 2: 24, 3: 10}

federated clients:
  Cedars-Sinai     n=  78  severity-labelled= 41
  Cincinnati       n=  89  severity-labelled= 39
  MGH              n=  60  severity-labelled= 28
  MGH Pediatrics   n=  24  severity-labelled= 10


## Experiment 1 — Config A across 3 seeds

Region-normalised features, 4 severity levels, 5 folds x 5 rounds x 50 epochs.
**This is the headline result.** Seeds 42/7/123 — a single seed was not reproducible (QWK ranged 0.18-0.31), so all reported numbers are seed-averaged.

In [10]:
#config A
from run_severity_fl import run_federated_severity, save_results
import numpy as np

runs = {}
for sd in [42, 7, 123]:
    print(f"\n{'#'*60}\nSEED {sd}\n{'#'*60}")
    res, sites = run_federated_severity(
        data, GATCrohnModel, NNwrapper, adj, geneList, DEVICE,
        n_folds=5, num_rounds=5, epochs_per_client=50,
        lam=0.5, region_zscore=True, seed=sd
    )
    runs[sd] = (res, sites)
    save_results((res, sites), tag=f"regionz_seed{sd}")

print("\n=== ACROSS SEEDS ===")
for k in ["mcc", "auc", "sev_spearman", "sev_qwk", "sev_mae_levels"]:
    per_seed = [np.mean([f[k] for f in r[0] if k in f and f[k] == f[k]])
                for r in runs.values()]
    print(f"  {k:18s} {np.mean(per_seed):.4f} +/- {np.std(per_seed):.4f}   {np.round(per_seed,3)}")


############################################################
SEED 42
############################################################

FOLD 1/5   train=199  test=52
  clients: {'Cedars-Sinai': '64(41 sev)', 'Cincinnati': '69(29 sev)', 'MGH': '48(23 sev)', 'MGH Pediatrics': '18(4 sev)'}
  round 1: MCC=-0.168 AUC=0.435 | rho=0.010 QWK=0.008 MAE=0.90 (n=21)
  round 2: MCC=0.041 AUC=0.504 | rho=0.037 QWK=0.007 MAE=0.90 (n=21)
  round 3: MCC=-0.102 AUC=0.507 | rho=0.037 QWK=0.007 MAE=0.90 (n=21)
  round 4: MCC=-0.023 AUC=0.478 | rho=0.159 QWK=0.123 MAE=0.81 (n=21)
  round 5: MCC=-0.103 AUC=0.519 | rho=0.076 QWK=0.044 MAE=0.90 (n=21)
  BEST -> MCC=-0.023 QWK=0.123

FOLD 2/5   train=204  test=47
  clients: {'Cedars-Sinai': '60(32 sev)', 'Cincinnati': '78(31 sev)', 'MGH': '48(26 sev)', 'MGH Pediatrics': '18(8 sev)'}
  round 1: MCC=0.277 AUC=0.645 | rho=0.352 QWK=0.220 MAE=0.71 (n=21)
  round 2: MCC=0.191 AUC=0.645 | rho=0.436 QWK=0.387 MAE=0.62 (n=21)
  round 3: MCC=0.276 AUC=0.665 | rho=0.398 QW

## Experiment 2 — Severity-specific gene attribution (XAI)

Gradient attribution on each head separately, then contrasted. Identifies genes that drive *severity* rather than *diagnosis*. Stability measured across 3 seeds; only genes in the top-50 for all seeds are reportable.

In [20]:
#severity xai
from severity_xai import severity_gene_importance, save_gene_report, stability_across_seeds
from run_severity_fl import participant_folds, _set_params
from hmp2_loader import make_node_features
import numpy as np

dfs = []
for sd in [42, 7, 123]:
    tr, te = participant_folds(data["labels"], 5, sd)[0]
    X = make_node_features(data["raw_counts"], train_idx=tr,
                           region=data["region"], region_zscore=True)
    net = GATCrohnModel(X.shape[2], X.shape[1], adj, geneList).to(DEVICE)
    w = NNwrapper(net)
    w.fit([X[i] for i in tr], [data["Y"][i] for i in tr],
          epochs=50, batch_size=4, lam=0.5, silent=True)
    dfs.append(severity_gene_importance(net, X, data["Y"], geneList, DEVICE))
    del net, w; t.cuda.empty_cache()

save_gene_report(dfs[0])
print("\n=== STABILITY ===")
st = stability_across_seeds(dfs, top_n=50)
print({k: v for k, v in st.items() if k != "in_all_seeds"})
print("genes in top-50 for ALL seeds:", st["in_all_seeds"])

=== TOP 50 GENES BY SEVERITY IMPORTANCE ===
    gene  sev_importance  diag_rank  contrast
    MT1X          1.1926          5    2.9453
  CDKN2A          0.7716        103    4.1920
   NPSR1          0.7517         17    2.0077
  CYP2A6          0.7336         58    3.2256
   ZPBP2          0.6969        146    3.9484
     DDO          0.6941        105    3.6267
   CLDN1          0.6821         15    1.0520
    MT1A          0.6239        101    3.0764
    PER3          0.5941         31    1.5792
    MT1H          0.5815         24    1.3151
      F2          0.5794         62    2.2254
     DAO          0.5709        354    3.6101
     ABO          0.5566          1   -7.0757
    GRM8          0.5325         21    0.8746
   PDSS2          0.5319        214    2.9658
     IL2          0.5172         29    0.9316
  COL8A2          0.5145         33    1.1593
    MT1M          0.5096         45    1.3333
   CLCA2          0.5005          3   -3.0329
 SLC23A1          0.4677        277 

## Experiment 3 — Differential privacy (top-100 genes)

Final DP attempt. Gene subsetting cuts the trunk from 708k to 102k params (sqrt(d) 842 -> 320) while keeping per-gene weighting, unlike pooling.

Gate run (no DP) held utility; the DP run did not.

In [ ]:
#no gate, DP
from run_severity_fl import run_federated_severity, save_results

res100, s100 = run_federated_severity(
    data, GATCrohnModel, NNwrapper, adj, geneList, DEVICE,
    n_folds=5, num_rounds=5, epochs_per_client=50, lam=0.5,
    region_zscore=True, top_k_genes=100
)
save_results((res100, s100), tag="top100_nodp")

In [11]:
# the actual DP test
from run_severity_fl import run_federated_severity, save_results
from severity_model import compute_epsilon

res_dp, s_dp = run_federated_severity(
    data, GATCrohnModel, NNwrapper, adj, geneList, DEVICE,
    n_folds=5, num_rounds=3, epochs_per_client=30, lam=0.5,
    region_zscore=True, top_k_genes=100, batch_size=16
)
save_results((res_dp, s_dp), tag="top100_dp_sigma15")
print("epsilon:", round(compute_epsilon(48, 16, 30, 3, 1.5), 2))


FOLD 1/5   train=199  test=52
  gene subset: 100/691 genes
  clients: {'Cedars-Sinai': '64(41 sev)', 'Cincinnati': '69(29 sev)', 'MGH': '48(23 sev)', 'MGH Pediatrics': '18(4 sev)'}
  round 1: MCC=0.022 AUC=0.518 | rho=0.000 QWK=0.000 MAE=0.76 (n=21)
  round 2: MCC=0.000 AUC=0.496 | rho=0.000 QWK=0.000 MAE=0.76 (n=21)
  round 3: MCC=0.000 AUC=0.518 | rho=0.000 QWK=0.000 MAE=0.76 (n=21)
  BEST -> MCC=0.022 QWK=0.000

FOLD 2/5   train=204  test=47
  clients: {'Cedars-Sinai': '60(32 sev)', 'Cincinnati': '78(31 sev)', 'MGH': '48(26 sev)', 'MGH Pediatrics': '18(8 sev)'}
  round 1: MCC=0.144 AUC=0.540 | rho=0.000 QWK=0.000 MAE=0.95 (n=21)
  round 2: MCC=0.144 AUC=0.428 | rho=0.000 QWK=0.000 MAE=0.95 (n=21)
  round 3: MCC=0.000 AUC=0.536 | rho=0.000 QWK=0.000 MAE=0.95 (n=21)
  BEST -> MCC=0.144 QWK=0.000

FOLD 3/5   train=197  test=54
  clients: {'Cedars-Sinai': '66(32 sev)', 'Cincinnati': '74(36 sev)', 'MGH': '41(15 sev)', 'MGH Pediatrics': '16(10 sev)'}
  round 1: MCC=-0.154 AUC=0.529 | rho

## Supporting cells

Region-feature ablations (A/B/C), centralized baseline, Non-IID site table, and the zero-fill check. Not all have stored outputs.

In [ ]:
import pandas as pd
counts = pd.read_csv("host_tx_counts.tsv", sep="\t", index_col=0, nrows=0)
present = set(pd.read_csv("host_tx_counts.tsv", sep="\t", index_col=0,
                          usecols=[0]).index)

stable = ['ABO','CDKN2A','CLDN1','COL8A2','DAO','DDO','F2','GSDMA','IGHG1','IL2',
          'MEFV','MICA','MT1A','MT1H','MT1M','MT1X','NLRP12','PER3','PTGS2','ZPBP2']
missing = [g for g in stable if g not in present]
print("zero-filled among stable genes:", missing if missing else "none — all real")

In [ ]:
#config A
from run_severity_fl import run_federated_severity, save_results
import numpy as np

runs = {}
for sd in [42, 7, 123]:
    print(f"\n{'#'*60}\nSEED {sd}\n{'#'*60}")
    res, sites = run_federated_severity(
        data, GATCrohnModel, NNwrapper, adj, geneList, DEVICE,
        n_folds=5, num_rounds=5, epochs_per_client=50,
        lam=0.5, region_zscore=True, seed=sd
    )
    runs[sd] = (res, sites)
    save_results((res, sites), tag=f"regionz_seed{sd}")

print("\n=== ACROSS SEEDS ===")
for k in ["mcc", "auc", "sev_spearman", "sev_qwk", "sev_mae_levels"]:
    per_seed = [np.mean([f[k] for f in r[0] if k in f and f[k] == f[k]])
                for r in runs.values()]
    print(f"  {k:18s} {np.mean(per_seed):.4f} +/- {np.std(per_seed):.4f}   {np.round(per_seed,3)}")

In [ ]:
#Cell A
from run_severity_fl import run_federated_severity, save_results
KW = dict(n_folds=5, num_rounds=5, epochs_per_client=50, lam=0.5)

resA = run_federated_severity(data, GATCrohnModel, NNwrapper, adj, geneList,
                              DEVICE, region_zscore=True, **KW)
save_results(resA, tag="regionz_v2")

In [ ]:
#Cell B
resB = run_federated_severity(data, GATCrohnModel, NNwrapper, adj, geneList,
                              DEVICE, region_flag=True, **KW)
save_results(resB, tag="regionflag")

In [ ]:
#Cell C
resC = run_federated_severity(data, GATCrohnModel, NNwrapper, adj, geneList,
                              DEVICE, region_zscore=True, region_flag=True, **KW)
save_results(resC, tag="regionboth")

In [ ]:
# Cell 11 — Centralized baseline, matched to config A
from run_severity_fl import run_centralized_baseline

res_central = run_centralized_baseline(
    data, GATCrohnModel, NNwrapper, adj, geneList, DEVICE,
    n_folds=5, epochs=100, lam=0.5, region_zscore=True
)

In [ ]:
# Cell 12 — Non-IID severity distribution across the real sites
# This figure stands on its own regardless of how the severity head performs.
import pandas as pd

lab = data["labels"]
cd  = lab[lab.sev_mask == 1]
tab = pd.crosstab(cd["site"], cd["y_sev"])
tab.columns = ["remission 0-2", "mild 3-6", "moderate 7-15", "severe 16+"][:tab.shape[1]]
print(tab.to_string())
print("\nrow % (severity mix per site):")
print((tab.div(tab.sum(axis=1), axis=0) * 100).round(1).to_string())

os.makedirs("/kaggle/working/results", exist_ok=True)
tab.to_csv("/kaggle/working/results/site_severity_distribution.csv")
print("\nsaved -> /kaggle/working/results/site_severity_distribution.csv")